# generator-loss-fool-discriminator — ex1: generator BCE loss with target=1 on the fake batch

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `generator-loss-fool-discriminator`. Running the final beacon cell reports progress against the `GAN: Generator loss to fool D` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Generator loss to fool D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`generator-loss-fool-discriminator`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "generator-loss-fool-discriminator"
DD_SUBTOPIC = "GAN: Generator loss to fool D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Generator loss (fool the discriminator) — quick refresher

The generator's loss is BCE on D's verdict for the fake batch, with target 1 — i.e. the generator WANTS D to call its fakes real:

```python
fakes = G(noise)
d_pred = D(fakes)
loss_G = F.binary_cross_entropy(d_pred, t.ones_like(d_pred))
```

**Note the asymmetry vs the D loss.** When D trains, the fake target is 0. When G trains, the fake target is 1 — same fakes, opposite label. That's the adversarial part.

**Why not `-loss_D_fake` (the math-paper form).** Goodfellow's original `min_G max_D` formulation gives `G_loss = log(1 - D(G(z)))` — but that gradient vanishes when D is confident the fake is fake (which is most of training). The flipped-target BCE — `-log(D(G(z)))` — gives strong gradient exactly when G needs it. This is the 'non-saturating loss' from the paper.

**Keep D frozen here.** Only G's parameters get gradient on this loss in the standard GAN training loop. The framework handles that via the optimizer split — `optim_G.step()` only touches G's params.

### Exercise 1 — generator BCE loss with target=1 on the fake batch

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `F.binary_cross_entropy(d_pred_fake, t.ones_like(...))` to compute the non-saturating generator loss — the loss that makes G push D's verdict toward 1.
> Keywords: gan, generator-loss, non-saturating-bce, ones-like
> ```

**KCs targeted:** `generator-bce-target-1`, `non-saturating-formulation`

Implement `ex1_generator_loss(d_pred_fake_via_g)`. The non-saturating generator loss from Goodfellow 2014:

1. `d_pred_fake_via_g` is D's probability output on the CURRENT generator's fakes — shape `(B,)`, values in `[0, 1]`.
2. Build the targets: `targets = t.ones_like(d_pred_fake_via_g)` — the generator WANTS D to output 1 on its fakes.
3. Return `F.binary_cross_entropy(d_pred_fake_via_g, targets)` — a scalar.

That's it — three lines. The subtlety is the asymmetry:
- D's loss uses target=0 for fakes ('fake means 0').
- G's loss uses target=1 for fakes ('I want D to think fake is real').

Same fakes, opposite labels — that's the adversarial signal.

Input: `d_pred_fake_via_g` — `(B,)` float tensor in `(0, 1)`.
Output: scalar tensor.

The visualization plots G loss as a function of D's confidence on the fake batch — monotonically decreasing (G is happiest when D is fooled to confidence 1).

In [ ]:
def ex1_generator_loss(d_pred_fake_via_g: Tensor) -> Tensor:
    """G wants D to predict 1 on fakes → BCE(d_pred, ones)."""
    raise NotImplementedError()


def _test_ex1():
    import torch.nn.functional as F
    import math

    # G fooled D completely — D outputs 1 on fakes → G loss → 0.
    fooled = t.full((8,), 0.9999)
    loss_fooled = ex1_generator_loss(fooled)
    assert loss_fooled.dim() == 0, 'loss must be a scalar'
    assert loss_fooled.item() < 0.001, f'fully-fooled D should give G loss ~0, got {loss_fooled.item():.5f}'

    # G failed completely — D outputs 0 on fakes → G loss → large.
    failed = t.full((8,), 0.0001)
    loss_failed = ex1_generator_loss(failed)
    assert loss_failed.item() > 5.0, f'fully-detected fakes should give large G loss, got {loss_failed.item():.4f}'

    # Coin-flip D — G loss = log(2) ≈ 0.6931.
    coin = t.full((8,), 0.5)
    loss_coin = ex1_generator_loss(coin)
    expected_coin = math.log(2)
    assert abs(loss_coin.item() - expected_coin) < 1e-4, f'coin-flip G loss expected {expected_coin:.4f}, got {loss_coin.item():.4f}'

    # Numerical match against reference.
    t.manual_seed(0)
    preds = t.rand(16) * 0.8 + 0.1
    got = ex1_generator_loss(preds)
    expected = F.binary_cross_entropy(preds, t.ones_like(preds))
    assert t.allclose(got, expected, atol=1e-6), f'numerical mismatch: {got.item()} vs {expected.item()}'

    # Gradient pushes D-prob UP (i.e. grad on d_pred is negative).
    p = t.full((4,), 0.5, requires_grad=True)
    ex1_generator_loss(p).backward()
    # d/dp BCE(p, 1) = -1/p < 0
    assert (p.grad < 0).all(), 'gradient should be negative (push d_pred toward 1)'

    # --- Visualization: G loss vs D's confidence on fakes ---
    ps = t.linspace(0.01, 0.99, 99)
    losses = [ex1_generator_loss(t.full((4,), p.item())).item() for p in ps]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ps.numpy(), losses, color='seagreen', lw=2)
    ax.axvline(1.0, color='red', ls=':', label='G goal: D(fake)=1')
    ax.set_xlabel('D probability on fakes'); ax.set_ylabel('generator loss')
    ax.set_title('non-saturating G loss — monotonically decreasing in D(fake)')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_generator_loss(d_pred_fake_via_g: Tensor) -> Tensor:
    import torch.nn.functional as F
    targets = t.ones_like(d_pred_fake_via_g)
    return F.binary_cross_entropy(d_pred_fake_via_g, targets)
```

**Non-saturating vs saturating.** The paper-original `min_G log(1 - D(G(z)))` saturates — when D is confident the fake is fake (D(fake) ≈ 0), `log(1 - D(fake)) ≈ log(1) ≈ 0` and the gradient w.r.t. G vanishes. The flipped target `BCE(D(fake), 1) = -log(D(fake))` instead goes to infinity in the same regime — strong gradient when G needs it most. Goodfellow's own footnote.

**G only sees D's verdict, not D's params.** The loss depends on `d_pred_fake_via_g` (a tensor produced by D), so autograd would normally try to flow gradient through D too. The training loop handles that by stepping ONLY `optim_G` after computing G's loss — `optim_G.param_groups` contain only G's parameters.

**One-liner is correct.** No need for ones-tensor construction logic — `ones_like` gives you the right shape/dtype/device in a single token. The whole loss really is three meaningful lines.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()